# Frozen safety prompt sets and model list — `plain_harmful` demo

This notebook is a runnable slice of **`data.py`**, the offline build script behind
`full_data_out.json`: **8 datasets / 2,113 rows**, every row tagged
`metadata_fold = <dataset name>`, with row schema
`{input, output, metadata_fold, metadata_uid, metadata_block_version, metadata_meta{...}}`.

The full build reads pinned local copies of 15 upstream sources (`temp/datasets/`), so it
cannot run here. What *can* run — and is what actually makes the corpus **frozen** — are the
deterministic derivations `data.py` applies on top of that raw text. This notebook replays
them against a shipped slice of the delivered corpus and re-checks that they reproduce the
frozen fields **exactly**:

1. `norm()` / `uid()` — the content-addressed row id. Re-derived from `input` alone and
   compared byte-for-byte against the shipped `metadata_uid`.
2. `guess_category()` — the 12-rule keyword classifier that gives AdvBench its categories
   (AdvBench ships none; JBB-Behaviors ships its own, which are kept and *not* overwritten).
3. `near_dup_keep()` — the greedy char-ngram TF-IDF near-duplicate filter used to dedupe the
   AdvBench + JBB union into 594 rows.
4. `sha256_rows()` + the build's assertion battery — 27 assertions ship in
   `metadata.assertions`; the ones that apply to this slice are re-run live.

**Data**: `mini_demo_data.json` — a stratified **100-row** sample of the 594-row
`plain_harmful` block (AdvBench + JBB-Behaviors union), round-robin over
(`meta.category` x `meta.in_core80`) so both the 80-row 10-category stratified core and the
wider pool are represented. `meta.target` carries the affirmative prefix used by GCG-style
attacks.

> Licence note carried from the corpus: `plain_harmful` is MIT (AdvBench, JBB-Behaviors).
> Two *other* blocks — `harmless_dynamics` and the `layer_contrast` benign half — are
> CC-BY-NC-4.0, **non-commercial**. This demo touches neither.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT pre-installed on Colab, always install (data.py logs through it)
_pip('loguru==0.7.3')

# numpy, pandas, scikit-learn, matplotlib — pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3',
         'matplotlib==3.10.0')

## Imports

The import block is `data.py`'s own, minus the filesystem/CLI bits that only make sense
inside the offline build (`Path`, `sys.argv` handling), plus `matplotlib` for the final
figure. `logger` is configured to stdout exactly as the script does it, so the assertion
log below is formatted the same way the real build logs it.

In [ ]:
from __future__ import annotations

import base64
import hashlib
import json
import re
import sys
from collections import Counter
from datetime import datetime, timezone

import pandas as pd
from loguru import logger
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

BLOCK_VERSION = "1.0.0"

## Load the data

`mini_demo_data.json` is fetched from GitHub, with a local file as fallback so the notebook
runs both in Colab and from a checkout.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-c9546e-rating-model-safety-in-eighty-forward-pa/main/round-1/dataset-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()

# the demo slice ships one dataset, but the same envelope also holds the full 8-dataset
# corpus, so select the block by name rather than by position.
BLOCK = "plain_harmful"
block = next(d for d in data["datasets"] if d["dataset"] == BLOCK)

meta = data["metadata"]
print(meta["name"])
print(f"  datasets in file   : {[d['dataset'] for d in data['datasets']]}")
print(f"  rows in {BLOCK}: {len(block['examples'])}")
print(f"  full block rows    : {meta.get('full_block_n_rows', meta.get('blocks', {}).get(BLOCK, 'n/a'))}")
print(f"  corpus version     : {meta['corpus_version']}   retrieved {meta['retrieved_utc']}")

## Config

Every tunable of this demo lives here. `N_ROWS` is the only scale knob; the rest are the
**exact** constants `data.py` uses, kept as variables so they are visible and changeable.

In [ ]:
# --- scale knob ---------------------------------------------------------------
N_ROWS = 100              # rows of plain_harmful to replay = every row this slice ships.
                          # The full block in full_data_out.json is 594 rows; point
                          # load_data() at it and set N_ROWS = None to replay all of them
                          # (the checks below are the same, they just take ~10x longer in
                          # the O(n^2) cosine step).

# --- constants copied from data.py (do not change to reproduce the frozen corpus) ---
DEDUP_THRESHOLD = 0.90    # data.py: near_dup_keep(texts, 0.90) for the B3 harmful union
DISJOINT_THRESHOLD = 0.85 # data.py: B5 cosine-disjointness assertion threshold
TFIDF_ANALYZER = "char_wb"
TFIDF_NGRAM_RANGE = (3, 5)
TFIDF_MIN_DF = 1
CORE80_FLOOR = 80         # SUBSET_FLOORS["plain_harmful in_core80"]
PLAIN_HARMFUL_FLOOR = 400 # FLOORS["plain_harmful"] (applies to the full block)
TOP_K_CATEGORIES = 14     # how many categories to show in the final chart

## Helpers, copied verbatim from `data.py`

`norm` -> `uid` is the whole content-addressing scheme: lowercase, strip everything that is
not `[a-z0-9 ]`, collapse, then SHA-1 of `"{block}||{normalised text}"` truncated to 16 hex
chars. `row()` is the single row constructor every block goes through, which is why all
2,113 rows share one schema. `sha256_rows` is the per-block digest recorded in
`metadata.manifest[<block>].sha256`.

In [ ]:
def norm(t: str) -> str:
    return re.sub(r"[^a-z0-9 ]+", " ", (t or "").lower()).strip()


def uid(block: str, text: str) -> str:
    return hashlib.sha1(f"{block}||{norm(text)}".encode()).hexdigest()[:16]


def sha256_rows(rows: list[dict]) -> str:
    return hashlib.sha256(json.dumps(rows, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def row(block: str, inp: str, out: str, meta: dict, key: str | None = None) -> dict:
    return {
        "input": inp,
        "output": out,
        "metadata_fold": block,
        "metadata_uid": uid(block, key if key is not None else inp),
        "metadata_block_version": BLOCK_VERSION,
        "metadata_meta": meta,
    }


def near_dup_keep(texts: list[str], thresh: float) -> list[int]:
    '''Greedy near-dup removal; returns indices to KEEP (first occurrence wins).'''
    if len(texts) < 2:
        return list(range(len(texts)))
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1).fit_transform(texts)
    keep: list[int] = [0]
    for i in range(1, len(texts)):
        if cosine_similarity(vec[i], vec[keep]).max() < thresh:
            keep.append(i)
    return keep

## The rows

Pull the single shipped block out of the corpus envelope and cut it to `N_ROWS`. The rows
arrive sorted by `metadata_uid`, which is how `data.py` emits every block
(`sorted(rows_, key=lambda r: r["metadata_uid"])`) — that sort is part of what makes the
per-block `sha256` stable.

In [ ]:
rows = block["examples"] if N_ROWS is None else block["examples"][:N_ROWS]

df = pd.DataFrame([{
    "uid": r["metadata_uid"],
    "input": r["input"],
    "target": r["metadata_meta"]["target"],
    "category": r["metadata_meta"]["category"],
    "category_source": r["metadata_meta"]["category_source"],
    "in_core80": r["metadata_meta"]["in_core80"],
    "origin_dataset": r["metadata_meta"]["origin_dataset"],
    "license": r["metadata_meta"]["license"],
} for r in rows])

logger.info(f"{BLOCK}: {len(df)} rows, {df.category.nunique()} categories, "
            f"{int(df.in_core80.sum())} in_core80")
df.head(5)

## 1. Re-derive `metadata_uid` from `input` alone

The claim being tested: the corpus is content-addressed, so anyone holding the prompt text
can recompute the id without the build. Any mismatch would mean a row's id does not follow
from its content — i.e. the block is not reproducible.

The block digest is recomputed the same way, over the rows exactly as shipped.

In [ ]:
recomputed = [uid(BLOCK, r["input"]) for r in rows]
uid_matches = sum(1 for r, u in zip(rows, recomputed) if r["metadata_uid"] == u)
logger.info(f"uid re-derivation: {uid_matches}/{len(rows)} exact matches")

# a worked example of what norm() strips before hashing
ex = rows[0]
print(f"\ninput : {ex['input']!r}")
print(f"norm  : {norm(ex['input'])!r}")
print(f"uid   : {uid(BLOCK, ex['input'])}  (shipped: {ex['metadata_uid']})")

# row() rebuilds the full schema from scratch and lands on the same id
rebuilt = row(BLOCK, ex["input"], ex["output"], ex["metadata_meta"])
print(f"\nrow() rebuild identical to shipped row: {rebuilt == ex}")

slice_sha = sha256_rows(rows)
print(f"sha256 of this {len(rows)}-row slice: {slice_sha[:32]}...")

## 2. Replay `guess_category()`

AdvBench ships **no** category labels, so `data.py` assigns them with an ordered 12-rule
keyword classifier — first rule that matches wins, else `Other/Unclassified`. JBB-Behaviors
*does* ship categories, and those are kept verbatim; `meta.category_source` records which
path each row took. So the re-derivation is only expected to reproduce the rows whose
`category_source` is the keyword rule — the JBB rows are compared separately to show how far
an independent keyword vote lands from a human-curated label.

In [ ]:
CATEGORY_RULES = [
    ("Malware/Hacking", r"malware|virus|ransomware|hack|exploit|keylogger|botnet|ddos|worm|trojan|spyware|rootkit|sql injection|bypass.*(antivirus|security)|crack.*password"),
    ("Fraud/Deception", r"scam|phish|fraud|counterfeit|forge|fake (id|review|news)|ponzi|identity theft|launder|insider trading|impersonat"),
    ("Physical harm", r"bomb|explosive|weapon|gun|firearm|poison|chemical weapon|biological weapon|nerve agent|anthrax|shoot|stab|kill|murder|assassinat"),
    ("Drugs", r"\bdrug|meth|cocaine|heroin|fentanyl|opioid|narcotic|lsd|mdma|synthesi[sz]e .*(drug|substance)"),
    ("Disinformation", r"disinformation|misinformation|propaganda|conspiracy|fake news|election.*(fraud|manipulat)|deny.*(holocaust|climate)"),
    ("Harassment/Bullying", r"harass|bully|stalk|troll|doxx|threaten|intimidat|revenge"),
    ("Sexual/Adult content", r"sexual|porn|explicit|nude|erotic|escort|prostitut"),
    ("Privacy", r"surveil|spy on|track.*(person|someone)|steal.*(data|identity|personal)|facial recognition|private information|medical record"),
    ("Economic harm", r"gambling|payday loan|multi-level marketing|casino|pyramid scheme|tax evasion|embezzl"),
    ("Government decision-making", r"vote|voting|election|immigration|law enforcement|parole|criminal justice"),
    ("Expert advice", r"medical advice|legal advice|financial advice|diagnos|prescri|self-medicat"),
    ("Hate/Discrimination", r"racist|racial|sexist|hate speech|xenophob|homophob|slur|supremacis|ethnic"),
]


def guess_category(text: str) -> str:
    t = text.lower()
    for name, pat in CATEGORY_RULES:
        if re.search(pat, t):
            return name
    return "Other/Unclassified"


df["category_guess"] = [guess_category(t) for t in df["input"]]
df["category_agrees"] = df["category_guess"] == df["category"]

is_rule = df["category_source"].str.contains("keyword rule")
rule_acc = df.loc[is_rule, "category_agrees"].mean() if is_rule.any() else float("nan")
jbb_acc = df.loc[~is_rule, "category_agrees"].mean() if (~is_rule).any() else float("nan")
logger.info(f"keyword-rule rows reproduced : {int(df.loc[is_rule, 'category_agrees'].sum())}/{int(is_rule.sum())} ({rule_acc:.3f})")
logger.info(f"JBB-labelled rows agreed with : {int(df.loc[~is_rule, 'category_agrees'].sum())}/{int((~is_rule).sum())} ({jbb_acc:.3f})")

print("\ncategory_source breakdown:")
print(df["category_source"].value_counts().to_string())
print("\nrows where an independent keyword vote disagrees with the shipped label:")
print(df.loc[~df.category_agrees, ["category", "category_guess", "input"]].head(5).to_string(index=False, max_colwidth=70))

## 3. Replay `near_dup_keep()`

`plain_harmful` is the **deduped** AdvBench + JBB union. `near_dup_keep` is greedy and
first-occurrence-wins: walk the rows in order, keep a row only if its maximum char-ngram
TF-IDF cosine against everything kept so far is below the threshold. Re-running it on the
delivered rows should keep essentially all of them — the filter has already been applied
upstream, so anything it now drops is a residual near-duplicate that survived at the build's
threshold ordering.

The same vectoriser then gives the closest pair of distinct prompts in the slice. Note which
threshold that has to clear: the build's `0.85` is a **cross-block** criterion (it asserts
`layer_contrast`'s harmful half is disjoint from `plain_harmful`, and recorded `0.652`).
*Within* `plain_harmful`, the binding constraint is the dedup threshold `0.90` — pairs
between `0.85` and `0.90` are kept by construction, because AdvBench and JBB genuinely
contain distinct behaviours that share most of their wording. So the check below is
`max cosine < 0.90`, and `0.85` is drawn on the figure only as a reference line.

In [ ]:
texts = df["input"].tolist()

keep = near_dup_keep(texts, DEDUP_THRESHOLD)
logger.info(f"near_dup_keep(thresh={DEDUP_THRESHOLD}): keeps {len(keep)}/{len(texts)} rows")

vec = TfidfVectorizer(analyzer=TFIDF_ANALYZER, ngram_range=TFIDF_NGRAM_RANGE,
                      min_df=TFIDF_MIN_DF).fit_transform(texts)
sim = cosine_similarity(vec)
import numpy as np
np.fill_diagonal(sim, 0.0)
max_cos = float(sim.max())
i, j = np.unravel_index(sim.argmax(), sim.shape)
logger.info(f"closest distinct pair in slice: cosine={max_cos:.4f} "
            f"(dedup thresh {DEDUP_THRESHOLD}; cross-block criterion is {DISJOINT_THRESHOLD})")
print(f"\n  [{i}] {texts[i]}")
print(f"  [{j}] {texts[j]}")

## 4. Re-run the build's assertions

`data.py` gathers every check into one `checks` list of `(name, passed, detail)` and refuses
to write the output if any fails; the whole list ships in `metadata.assertions`. Below is the
subset that is meaningful on a single-block slice, evaluated live and logged in the build's
own `[PASS]/[FAIL]` format, followed by the shipped verdicts for the same block.

In [ ]:
checks: list[tuple[str, bool, str]] = []

checks.append(("uid re-derives from input", uid_matches == len(rows), f"{uid_matches}/{len(rows)}"))
checks.append(("no duplicate uids", len(set(df.uid)) == len(df), f"{len(df) - len(set(df.uid))} dupes"))
checks.append(("every row tagged metadata_fold", all(r["metadata_fold"] == BLOCK for r in rows), BLOCK))
checks.append(("every row carries block_version", all(r["metadata_block_version"] == BLOCK_VERSION for r in rows), BLOCK_VERSION))
checks.append(("every row has an affirmative meta.target", all(r["metadata_meta"]["target"] for r in rows),
               f"{sum(1 for r in rows if not r['metadata_meta']['target'])} empty"))
checks.append(("core80 stratified over >= 8 categories",
               df.loc[df.in_core80, "category"].nunique() >= 8,
               f"{df.loc[df.in_core80, 'category'].nunique()} categories"))
checks.append(("keyword-rule categories reproduce exactly",
               bool(df.loc[is_rule, "category_agrees"].all()),
               f"{rule_acc:.3f}"))
checks.append((f"near-dup free at thresh {DEDUP_THRESHOLD}", len(keep) == len(texts), f"{len(keep)}/{len(texts)}"))
checks.append((f"no within-block pair at/above dedup thresh {DEDUP_THRESHOLD}",
               max_cos < DEDUP_THRESHOLD,
               f"max_cos={max_cos:.4f} (cross-block criterion {DISJOINT_THRESHOLD} is a "
               f"different, tighter test on a different pair of blocks)"))
checks.append(("row schema is the frozen 6-key schema",
               all(set(r) == {"input", "output", "metadata_fold", "metadata_uid",
                              "metadata_block_version", "metadata_meta"} for r in rows),
               f"{sorted(rows[0])}"))

for name, ok_, detail in checks:
    logger.info(f"  [{'PASS' if ok_ else 'FAIL'}] {name:<52} {detail}")
failed = [c[0] for c in checks if not c[1]]
logger.info(f"{len(checks) - len(failed)}/{len(checks)} live checks passed")

print("\nshipped build assertions touching this block:")
for a in data["metadata"].get("assertions", []):
    if BLOCK in a["check"] or "uid" in a["check"]:
        print(f"  [{'PASS' if a['passed'] else 'FAIL'}] {a['check']:<52} {a['detail']}")

## Results

Left: the category stratification of the slice, split by `meta.in_core80` — the 80-row core
is deliberately flat across categories, the wider pool is not. Right: the distribution of
pairwise prompt similarity, with the `0.90` dedup threshold that actually binds within this
block and the `0.85` cross-block criterion both marked. The table above them is the live
assertion battery.

In [ ]:
summary = pd.DataFrame(
    [{"check": n, "result": "PASS" if ok_ else "FAIL", "detail": d} for n, ok_, d in checks]
)
print(summary.to_string(index=False, max_colwidth=60))
print(f"\nrows={len(df)}  categories={df.category.nunique()}  "
       f"in_core80={int(df.in_core80.sum())}  uid_matches={uid_matches}  max_cos={max_cos:.4f}")

order = df["category"].value_counts().index[:TOP_K_CATEGORIES][::-1]
core = df[df.in_core80]["category"].value_counts().reindex(order).fillna(0)
pool = df[~df.in_core80]["category"].value_counts().reindex(order).fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ax = axes[0]
ax.barh(order, core, color="#c0392b", label="in_core80 (stratified core)")
ax.barh(order, pool, left=core, color="#95a5a6", label="wider pool")
ax.set_xlabel("rows")
ax.set_title(f"{BLOCK}: category x in_core80  (n={len(df)})")
ax.legend(loc="lower right", fontsize=9)

ax = axes[1]
tri = sim[np.triu_indices_from(sim, k=1)]
ax.hist(tri, bins=60, color="#2c7fb8")
ax.axvline(DEDUP_THRESHOLD, color="#c0392b", ls="--",
           label=f"dedup threshold {DEDUP_THRESHOLD} (binding here)")
ax.axvline(DISJOINT_THRESHOLD, color="#7f8c8d", ls="-.",
           label=f"cross-block criterion {DISJOINT_THRESHOLD} (other blocks)")
ax.axvline(max_cos, color="#e67e22", ls=":", label=f"observed max {max_cos:.3f}")
ax.set_xlabel("pairwise char-ngram TF-IDF cosine")
ax.set_ylabel("prompt pairs")
ax.set_yscale("log")
ax.set_title("prompt-to-prompt similarity within the slice")
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()